In [1]:
from langchain.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.memory import ConversationBufferMemory
from langchain.llms import OpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.chat_models import ChatOpenAI

import openai
import config
openai.api_key = config.apiKey

In [2]:
url = "https://365datascience.com/courses"

In [3]:
loader = WebBaseLoader(url)

In [4]:
raw_docs = loader.load() #load the text and convert to langchain doc
raw_docs[0].page_content[:2000]

'\n\n\nOnline Data Science Courses and Certification – 365 Data Science\n\n\n\n  \n          Courses\n         \n          Learning Paths  Learning Paths\n             \n                Career Paths  Career Paths\n                   See all  Career Paths See all  \n                        Data analyst\n                      \n                        Data scientist\n                      \n                        Business analyst\n                      \n                        Senior data analyst\n                      \n                        Senior data scientist\n                      \n                        Tableau developer\n                      \n                        Power BI developer\n                      \n                        Data engineer\n                      \n                        Machine learning scientist\n                      \n                        AI engineer\n                      \n                        AI agent engineer\n                       \

In [5]:
text_splitter = RecursiveCharacterTextSplitter() #pass only smallest relevant pieces of text to the llm
docs = text_splitter.split_documents(raw_docs)

In [6]:
embeddings = OpenAIEmbeddings(openai_api_key = config.apiKey,
                              openai_api_base="https://openrouter.ai/api/v1")

In [7]:
vectorStore = FAISS.from_documents(docs, embeddings)

In [8]:
memory = ConversationBufferMemory(memory_key = "chat_history", return_messages = True) # track in and out to hold conversation
# init convers retrival chain, combines chat history and q to a q, looks up rel docs in the retrieval and passes those docs and qs to the chain

In [9]:
qa = ConversationalRetrievalChain.from_llm(ChatOpenAI(
                        openai_api_key = config.apiKey,
                        openai_api_base="https://openrouter.ai/api/v1", 
                        model_name="openai/gpt-4",
                        temperature= 0),
                         vectorStore.as_retriever(), memory = memory)

In [10]:
query = "What is the next course to be uploaded on the 365Datascience platform?"

In [11]:
result = qa({"question": query})
result

{'question': 'What is the next course to be uploaded on the 365Datascience platform?',
 'chat_history': [HumanMessage(content='What is the next course to be uploaded on the 365Datascience platform?', additional_kwargs={}, example=False),
  AIMessage(content="The text doesn't provide information on what the next course to be uploaded on the 365 Data Science platform will be.", additional_kwargs={}, example=False)],
 'answer': "The text doesn't provide information on what the next course to be uploaded on the 365 Data Science platform will be."}

In [12]:
result["answer"]

"The text doesn't provide information on what the next course to be uploaded on the 365 Data Science platform will be."